In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
from libpysal import graph
from esda.moran import Moran

In [3]:
sac_variables ={"sacramento":'HH_INC',
                "elections":"pct_dem_12",
                "guerry":"Donatns"}

In [4]:
def compute_truth_autocorrelation_real(name, var_col, gdf):
    graph_path = f"graphs/real/{name}/g_true.parquet"
    g = graph.read_parquet(graph_path)
    wm = g.to_W()
    wm.transform = 'r'

    node_ids = list(wm.id_order)
    y = gdf.loc[node_ids][var_col].values   # .loc not .iloc

    mi = Moran(y, wm)
    global_result = {
        "dataset": name,
        "variable": var_col,
        "moran_i": mi.I, "moran_p": mi.p_sim, "moran_z": mi.z_sim,
    }

    return global_result

In [6]:
truth_global_results = []
global_dir = Path("results/autocorrelation/real/truth/global")
global_dir.mkdir(parents=True, exist_ok=True)

for name, var_col in sac_variables.items():
    src_path = f"data/real/{name}.parquet"
    gdf = gpd.read_parquet(src_path)

    global_result = compute_truth_autocorrelation_real(name, var_col, gdf)
    truth_global_results.append(global_result)

    print(f"Done: {name}")

pd.DataFrame(truth_global_results).to_parquet(global_dir / "global.parquet")

Done: sacramento
Done: elections
Done: guerry
